# Phase 2: Data Preprocessing & Event Grouping
## Test Notebook - LoneWolf Dataset

### Objective
Prepare forensic-ready structured event data by:
1. Joining LogFile events to MFT entries (via TargetFRN -> EntryNumber)
2. Joining UsnJrnl events to MFT entries (via FRN -> EntryNumber)
3. Normalizing timestamps (preserving nanoseconds, UTC)
4. Grouping events per file, ordered by time
5. Adding dataID column for dataset tracking

### Important Notes
- This phase does NOT apply detection logic (that is Phase 3)
- This phase does NOT label events
- All timestamps preserve nanosecond precision
- Output is event-level data, not file-level

### Input Files
- LoneWolf-LogFile.csv
- LoneWolf-MFT.csv
- LoneWolf-UsnJrnl.csv

### Output File
- grouped_events_LoneWolf.csv


In [1]:
# [Cell 2] Imports and Configuration

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

# Configuration
DATASET_NAME = "LoneWolf"
DATA_ID = "LoneWolf"

# Input paths
INPUT_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis/data/Phase 1: Raw Data Parsing/LoneWolf/LoneWolf")
LOGFILE_PATH = INPUT_DIR / "LoneWolf-LogFile.csv"
MFT_PATH = INPUT_DIR / "LoneWolf-MFT.csv"
USNJRNL_PATH = INPUT_DIR / "LoneWolf-UsnJrnl.csv"

# Output paths
OUTPUT_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis/data/Phase 2: Data Preprocessing & Event Grouping/Test Notebook Outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "grouped_events_LoneWolf.csv"

print(f"Dataset: {DATASET_NAME}")
print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


Dataset: LoneWolf
Input directory: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 1: Raw Data Parsing/LoneWolf/LoneWolf
Output directory: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 2: Data Preprocessing & Event Grouping/Test Notebook Outputs


## Step 1: Load Parsed CSV Files

Load all three artifacts from Phase 1 parsing output.

In [2]:
# [Cell 4] Load Parsed CSV Files

# Load MFT - this is our reference for file information
print("Loading MFT...")
df_mft = pd.read_csv(MFT_PATH, low_memory=False)
print(f"  MFT records: {len(df_mft):,}")

# Load LogFile
print("Loading LogFile...")
df_logfile = pd.read_csv(LOGFILE_PATH, low_memory=False)
print(f"  LogFile records: {len(df_logfile):,}")

# Load UsnJrnl
print("Loading UsnJrnl...")
df_usnjrnl = pd.read_csv(USNJRNL_PATH, low_memory=False)
print(f"  UsnJrnl records: {len(df_usnjrnl):,}")

print("\n--- Column Summary ---")
print(f"MFT columns: {list(df_mft.columns)}")
print(f"LogFile columns: {list(df_logfile.columns)}")
print(f"UsnJrnl columns: {list(df_usnjrnl.columns)}")


Loading MFT...
  MFT records: 142,960
Loading LogFile...
  LogFile records: 294,761
Loading UsnJrnl...
  UsnJrnl records: 352,849

--- Column Summary ---
MFT columns: ['EntryNumber', 'FileName', 'FilePath', 'IsActive', 'LSN', 'ParentFRN', '$SI-C', '$SI-M', '$SI-E', '$SI-A', '$FN-C', '$FN-M', '$FN-E', '$FN-A']
LogFile columns: ['LSN', 'RedoOP', 'UndoOP', 'RedoOPName', 'UndoOPName', 'RecordOffset', 'AttributeOffset', 'TargetVCN', 'TargetFRN', 'IsTimestampChange', 'Undo_$SI-C', 'Undo_$SI-M', 'Undo_$SI-E', 'Undo_$SI-A', 'Redo_$SI-C', 'Redo_$SI-M', 'Redo_$SI-E', 'Redo_$SI-A']
UsnJrnl columns: ['USN', 'FRN', 'ParentFRN', 'Timestamp', 'FileName', 'ReasonCode', 'ReasonFlags', 'SourceInfo', 'HasBasicInfoChange', 'HasClose', 'HasFileCreate']


## Step 2: Preprocess MFT Reference Table

Create a lookup table from MFT for joining with LogFile and UsnJrnl events.
- Key: EntryNumber (FRN)
- Values: FileName, FilePath, IsActive, $SI timestamps, $FN timestamps


In [3]:
# [Cell 6] Preprocess MFT Reference Table

# Create MFT lookup dictionary for efficient joining
# Key is EntryNumber which corresponds to FRN in other artifacts

mft_columns_to_keep = [
    'EntryNumber', 'FileName', 'FilePath', 'IsActive', 'LSN', 'ParentFRN',
    '$SI-C', '$SI-M', '$SI-E', '$SI-A',
    '$FN-C', '$FN-M', '$FN-E', '$FN-A'
]

df_mft_ref = df_mft[mft_columns_to_keep].copy()

# Rename EntryNumber to FileFRN for consistency
df_mft_ref = df_mft_ref.rename(columns={'EntryNumber': 'FileFRN'})

# Handle missing values in MFT
df_mft_ref['FileName'] = df_mft_ref['FileName'].fillna('')
df_mft_ref['FilePath'] = df_mft_ref['FilePath'].fillna('')

print(f"MFT reference table created: {len(df_mft_ref):,} entries")
print(f"Active files: {df_mft_ref['IsActive'].sum():,}")
print(f"Inactive files: {(~df_mft_ref['IsActive']).sum():,}")

# Show sample
print("\nMFT Reference Sample:")
df_mft_ref.head(3)


MFT reference table created: 142,960 entries
Active files: 139,886
Inactive files: 3,074

MFT Reference Sample:


,FileFRN,FileName,FilePath,IsActive,LSN,ParentFRN,$SI-C,$SI-M,$SI-E,$SI-A,$FN-C,$FN-M,$FN-E,$FN-A
0,0,$MFT,/$MFT,True,1462861135,5.0,2018-03-27 13:07:33.477048,2018-03-27 13:07:33.477048,2018-03-27 13:07:33.477048,2018-03-27 13:07:33.477048,2018-03-27 13:07:33.477048,2018-03-27 13:07:33.477048,2018-03-27 13:07:33.477048,2018-03-27 13:07:33.477048
1,1,$MFTMirr,/$MFTMirr,True,33559616,5.0,2018-03-27 13:07:33.477048,2018-03-27 13:07:33.477048,2018-03-27 13:07:33.477048,2018-03-27 13:07:33.477048,2018-03-27 13:07:33.477048,2018-03-27 13:07:33.477048,2018-03-27 13:07:33.477048,2018-03-27 13:07:33.477048
2,2,$LogFile,/$LogFile,True,33559686,5.0,2018-03-27 13:07:33.477048,2018-03-27 13:07:33.477048,2018-03-27 13:07:33.477048,2018-03-27 13:07:33.477048,2018-03-27 13:07:33.477048,2018-03-27 13:07:33.477048,2018-03-27 13:07:33.477048,2018-03-27 13:07:33.477048


## Step 3: Process LogFile Events

Process LogFile records and join with MFT to get file information.
- Join on: TargetFRN (LogFile) -> FileFRN (MFT)
- Focus on timestamp change events (IsTimestampChange = True) for forensic relevance
- Preserve all events for complete event sequencing


In [4]:
# [Cell 8] Process LogFile Events

# Clean TargetFRN - convert to integer for joining
df_logfile['TargetFRN_clean'] = pd.to_numeric(df_logfile['TargetFRN'], errors='coerce')
df_logfile['TargetFRN_clean'] = df_logfile['TargetFRN_clean'].fillna(-1).astype(int)

# Filter out records with no valid TargetFRN (system records)
df_logfile_valid = df_logfile[df_logfile['TargetFRN_clean'] >= 0].copy()

print(f"Total LogFile records: {len(df_logfile):,}")
print(f"Records with valid TargetFRN: {len(df_logfile_valid):,}")
print(f"Records without valid TargetFRN: {len(df_logfile) - len(df_logfile_valid):,}")

# Count timestamp change events
ts_change_count = df_logfile_valid['IsTimestampChange'].sum()
print(f"Timestamp change events: {ts_change_count:,}")

# Join LogFile with MFT reference
df_logfile_joined = df_logfile_valid.merge(
    df_mft_ref[['FileFRN', 'FileName', 'FilePath', 'IsActive']],
    left_on='TargetFRN_clean',
    right_on='FileFRN',
    how='left'
)

# Add event source identifier
df_logfile_joined['EventSource'] = 'LogFile'

# For LogFile events, use Redo_$SI-E as EventTimestamp when available
# This represents when the entry was modified
df_logfile_joined['EventTimestamp'] = df_logfile_joined['Redo_$SI-E']

# For non-timestamp-change events, EventTimestamp will be NaT
# We'll handle ordering by LSN for these

print(f"\nLogFile events after MFT join: {len(df_logfile_joined):,}")
print(f"Events matched to MFT: {df_logfile_joined['FileFRN'].notna().sum():,}")
print(f"Events not matched: {df_logfile_joined['FileFRN'].isna().sum():,}")


Total LogFile records: 294,761
Records with valid TargetFRN: 128,551
Records without valid TargetFRN: 166,210
Timestamp change events: 20,373

LogFile events after MFT join: 128,551
Events matched to MFT: 124,516
Events not matched: 4,035


## Step 4: Process UsnJrnl Events

Process UsnJrnl records and join with MFT to get file information.
- Join on: FRN (UsnJrnl) -> FileFRN (MFT)
- UsnJrnl has native Timestamp for event ordering


In [5]:
# [Cell 10] Process UsnJrnl Events

# UsnJrnl FRN is already the file reference number
df_usnjrnl['FRN_clean'] = pd.to_numeric(df_usnjrnl['FRN'], errors='coerce')
df_usnjrnl['FRN_clean'] = df_usnjrnl['FRN_clean'].fillna(-1).astype(int)

# Filter valid records
df_usnjrnl_valid = df_usnjrnl[df_usnjrnl['FRN_clean'] >= 0].copy()

print(f"Total UsnJrnl records: {len(df_usnjrnl):,}")
print(f"Records with valid FRN: {len(df_usnjrnl_valid):,}")

# Count detection-relevant patterns
basic_info_count = df_usnjrnl_valid['HasBasicInfoChange'].sum()
close_count = df_usnjrnl_valid['HasClose'].sum()
create_count = df_usnjrnl_valid['HasFileCreate'].sum()

print(f"\nDetection-relevant flags:")
print(f"  HasBasicInfoChange: {basic_info_count:,}")
print(f"  HasClose: {close_count:,}")
print(f"  HasFileCreate: {create_count:,}")

# Join UsnJrnl with MFT reference
df_usnjrnl_joined = df_usnjrnl_valid.merge(
    df_mft_ref[['FileFRN', 'FileName', 'FilePath', 'IsActive']],
    left_on='FRN_clean',
    right_on='FileFRN',
    how='left',
    suffixes=('_usn', '_mft')
)

# Use MFT filename if UsnJrnl filename is missing
df_usnjrnl_joined['FileName'] = df_usnjrnl_joined['FileName_usn'].fillna(df_usnjrnl_joined['FileName_mft'])
df_usnjrnl_joined = df_usnjrnl_joined.drop(columns=['FileName_usn', 'FileName_mft'])

# Add event source identifier
df_usnjrnl_joined['EventSource'] = 'UsnJrnl'

# Use native Timestamp as EventTimestamp
df_usnjrnl_joined['EventTimestamp'] = df_usnjrnl_joined['Timestamp']

print(f"\nUsnJrnl events after MFT join: {len(df_usnjrnl_joined):,}")
print(f"Events matched to MFT: {df_usnjrnl_joined['FileFRN'].notna().sum():,}")
print(f"Events not matched: {df_usnjrnl_joined['FileFRN'].isna().sum():,}")


Total UsnJrnl records: 352,849
Records with valid FRN: 352,849

Detection-relevant flags:
  HasBasicInfoChange: 42
  HasClose: 169,502
  HasFileCreate: 146,838

UsnJrnl events after MFT join: 352,849
Events matched to MFT: 348,301
Events not matched: 4,548


## Step 5: Normalize Event Schema

Create a unified event schema for both LogFile and UsnJrnl events.
This enables combined event sequencing per file.

### Unified Schema
| Column | Description | Source |
|--------|-------------|--------|
| dataID | Dataset identifier | Added |
| FileFRN | File Reference Number | Both |
| FileName | File name | MFT |
| FilePath | Full file path | MFT |
| EventSource | 'LogFile' or 'UsnJrnl' | Added |
| EventTimestamp | Event time for ordering | Both |
| LSN | Log Sequence Number | LogFile |
| USN | Update Sequence Number | UsnJrnl |
| ... | Source-specific columns | Varies |


In [6]:
# [Cell 12] Normalize LogFile Event Schema

logfile_columns = [
    # Identifiers
    'FileFRN', 'FileName', 'FilePath', 'EventSource', 'EventTimestamp',
    # LogFile specific
    'LSN', 'RedoOP', 'UndoOP', 'RedoOPName', 'UndoOPName',
    'RecordOffset', 'AttributeOffset', 'TargetVCN',
    'IsTimestampChange',
    # Undo timestamps (before state)
    'Undo_$SI-C', 'Undo_$SI-M', 'Undo_$SI-E', 'Undo_$SI-A',
    # Redo timestamps (after state)
    'Redo_$SI-C', 'Redo_$SI-M', 'Redo_$SI-E', 'Redo_$SI-A'
]

# Select and reorder columns
df_logfile_normalized = df_logfile_joined[logfile_columns].copy()

# Add placeholder columns for UsnJrnl-specific fields
df_logfile_normalized['USN'] = np.nan
df_logfile_normalized['ReasonCode'] = np.nan
df_logfile_normalized['ReasonFlags'] = ''
df_logfile_normalized['HasBasicInfoChange'] = False
df_logfile_normalized['HasClose'] = False
df_logfile_normalized['HasFileCreate'] = False

# Add dataID
df_logfile_normalized['dataID'] = DATA_ID

print(f"LogFile events normalized: {len(df_logfile_normalized):,}")
print(f"Columns: {len(df_logfile_normalized.columns)}")


LogFile events normalized: 128,551
Columns: 29


In [7]:
# [Cell 13] Normalize UsnJrnl Event Schema

# Select UsnJrnl columns
usnjrnl_base_columns = [
    'FileFRN', 'FileName', 'FilePath', 'EventSource', 'EventTimestamp',
    'USN', 'ReasonCode', 'ReasonFlags',
    'HasBasicInfoChange', 'HasClose', 'HasFileCreate'
]

df_usnjrnl_normalized = df_usnjrnl_joined[usnjrnl_base_columns].copy()

# Add placeholder columns for LogFile-specific fields
df_usnjrnl_normalized['LSN'] = np.nan
df_usnjrnl_normalized['RedoOP'] = np.nan
df_usnjrnl_normalized['UndoOP'] = np.nan
df_usnjrnl_normalized['RedoOPName'] = ''
df_usnjrnl_normalized['UndoOPName'] = ''
df_usnjrnl_normalized['RecordOffset'] = np.nan
df_usnjrnl_normalized['AttributeOffset'] = np.nan
df_usnjrnl_normalized['TargetVCN'] = np.nan
df_usnjrnl_normalized['IsTimestampChange'] = False

# Undo/Redo timestamps not applicable for UsnJrnl
df_usnjrnl_normalized['Undo_$SI-C'] = ''
df_usnjrnl_normalized['Undo_$SI-M'] = ''
df_usnjrnl_normalized['Undo_$SI-E'] = ''
df_usnjrnl_normalized['Undo_$SI-A'] = ''
df_usnjrnl_normalized['Redo_$SI-C'] = ''
df_usnjrnl_normalized['Redo_$SI-M'] = ''
df_usnjrnl_normalized['Redo_$SI-E'] = ''
df_usnjrnl_normalized['Redo_$SI-A'] = ''

# Add dataID
df_usnjrnl_normalized['dataID'] = DATA_ID

print(f"UsnJrnl events normalized: {len(df_usnjrnl_normalized):,}")
print(f"Columns: {len(df_usnjrnl_normalized.columns)}")


UsnJrnl events normalized: 352,849
Columns: 29


## Step 6: Combine Event Streams

Merge LogFile and UsnJrnl events into a single event stream.
Group by file (FileFRN) and order by EventTimestamp/LSN/USN.


In [8]:
# [Cell 15] Combine and Order Events

# Define final column order
final_columns = [
    # Identifiers
    'dataID', 'FileFRN', 'FileName', 'FilePath', 'EventSource', 'EventTimestamp',
    # Sequence numbers
    'LSN', 'USN',
    # LogFile fields
    'RedoOP', 'UndoOP', 'RedoOPName', 'UndoOPName',
    'RecordOffset', 'AttributeOffset', 'TargetVCN',
    'IsTimestampChange',
    # LogFile timestamps
    'Undo_$SI-C', 'Undo_$SI-M', 'Undo_$SI-E', 'Undo_$SI-A',
    'Redo_$SI-C', 'Redo_$SI-M', 'Redo_$SI-E', 'Redo_$SI-A',
    # UsnJrnl fields
    'ReasonCode', 'ReasonFlags',
    'HasBasicInfoChange', 'HasClose', 'HasFileCreate'
]

# Ensure both DataFrames have all columns
for col in final_columns:
    if col not in df_logfile_normalized.columns:
        df_logfile_normalized[col] = np.nan
    if col not in df_usnjrnl_normalized.columns:
        df_usnjrnl_normalized[col] = np.nan

# Reorder columns
df_logfile_final = df_logfile_normalized[final_columns]
df_usnjrnl_final = df_usnjrnl_normalized[final_columns]

# Combine event streams
df_combined = pd.concat([df_logfile_final, df_usnjrnl_final], ignore_index=True)

print(f"Combined events: {len(df_combined):,}")
print(f"  From LogFile: {len(df_logfile_final):,}")
print(f"  From UsnJrnl: {len(df_usnjrnl_final):,}")


Combined events: 481,400
  From LogFile: 128,551
  From UsnJrnl: 352,849


In [9]:
# [Cell 16] Group and Sort Events Per File

# Convert EventTimestamp to datetime for proper sorting
df_combined['EventTimestamp'] = pd.to_datetime(df_combined['EventTimestamp'], errors='coerce')

# Create sort keys:
# 1. Primary: FileFRN (group by file)
# 2. Secondary: EventTimestamp (chronological order)
# 3. Tertiary: LSN for LogFile events, USN for UsnJrnl events

# For events without EventTimestamp, use a sentinel date
sentinel_date = pd.Timestamp('1970-01-01')
df_combined['sort_timestamp'] = df_combined['EventTimestamp'].fillna(sentinel_date)

# Create composite sort key for sequence numbers
df_combined['sort_seq'] = df_combined['LSN'].fillna(0) + df_combined['USN'].fillna(0)

# Sort by FileFRN, then timestamp, then sequence number
df_grouped = df_combined.sort_values(
    by=['FileFRN', 'sort_timestamp', 'sort_seq'],
    ascending=[True, True, True]
).reset_index(drop=True)

# Remove temporary sort columns
df_grouped = df_grouped.drop(columns=['sort_timestamp', 'sort_seq'])

# Count unique files
unique_files = df_grouped['FileFRN'].nunique()
print(f"Total events after grouping: {len(df_grouped):,}")
print(f"Unique files with events: {unique_files:,}")

# Show event distribution per source
source_counts = df_grouped['EventSource'].value_counts()
print(f"\nEvents by source:")
for source, count in source_counts.items():
    print(f"  {source}: {count:,}")


Total events after grouping: 481,400
Unique files with events: 17,897

Events by source:
  UsnJrnl: 352,849
  LogFile: 128,551


## Step 7: Data Quality Checks

Validate the grouped events data before export.


In [10]:
# [Cell 18] Data Quality Validation

print("=== Data Quality Report ===\n")

# Check for missing FileFRN
missing_frn = df_grouped['FileFRN'].isna().sum()
print(f"Events with missing FileFRN: {missing_frn:,}")

# Check for missing EventTimestamp
missing_ts = df_grouped['EventTimestamp'].isna().sum()
print(f"Events with missing EventTimestamp: {missing_ts:,}")

# Check timestamp change events
ts_change_events = df_grouped[df_grouped['IsTimestampChange'] == True]
print(f"\nTimestamp change events (LogFile): {len(ts_change_events):,}")

# Check BASIC_INFO_CHANGE events (key for Oh et al. detection)
basic_info_events = df_grouped[df_grouped['HasBasicInfoChange'] == True]
print(f"BASIC_INFO_CHANGE events (UsnJrnl): {len(basic_info_events):,}")

# Files with timestamp change events
files_with_ts_change = ts_change_events['FileFRN'].nunique()
print(f"\nFiles with timestamp change events: {files_with_ts_change:,}")

# Files with BASIC_INFO_CHANGE
files_with_basic_info = basic_info_events['FileFRN'].nunique()
print(f"Files with BASIC_INFO_CHANGE: {files_with_basic_info:,}")

# Event count distribution per file
events_per_file = df_grouped.groupby('FileFRN').size()
print(f"\nEvents per file statistics:")
print(f"  Min: {events_per_file.min()}")
print(f"  Max: {events_per_file.max()}")
print(f"  Mean: {events_per_file.mean():.2f}")
print(f"  Median: {events_per_file.median():.0f}")


=== Data Quality Report ===

Events with missing FileFRN: 8,583
Events with missing EventTimestamp: 108,380

Timestamp change events (LogFile): 20,373
BASIC_INFO_CHANGE events (UsnJrnl): 42

Files with timestamp change events: 1,559
Files with BASIC_INFO_CHANGE: 21

Events per file statistics:
  Min: 1
  Max: 50138
  Mean: 26.42
  Median: 9


In [11]:
# [Cell 19] Preview Forensically Interesting Events

# Show sample of timestamp change events
print("=== Sample Timestamp Change Events (LogFile) ===\n")

ts_sample = ts_change_events[['FileFRN', 'FileName', 'EventTimestamp', 'LSN', 
                               'Undo_$SI-C', 'Redo_$SI-C', 
                               'Undo_$SI-M', 'Redo_$SI-M']].head(10)
print(ts_sample.to_string())

# Show sample of BASIC_INFO_CHANGE events
print("\n\n=== Sample BASIC_INFO_CHANGE Events (UsnJrnl) ===\n")

basic_sample = basic_info_events[['FileFRN', 'FileName', 'EventTimestamp', 'USN',
                                   'ReasonFlags', 'HasClose']].head(10)
print(basic_sample.to_string())


=== Sample Timestamp Change Events (LogFile) ===

      FileFRN                   FileName             EventTimestamp           LSN Undo_$SI-C Redo_$SI-C                  Undo_$SI-M                  Redo_$SI-M
3392     49.0                MputHistory 2018-04-06 03:40:11.978132  1.561354e+09        NaN        NaN  2018-03-27 21:45:23.210353  2018-04-06 03:40:11.978132
3393     50.0                         18 2018-04-06 03:40:11.972130  1.549547e+09        NaN        NaN  2018-03-28 13:45:49.509388  2018-04-06 03:40:11.972130
3463     51.0  HxCommAlwaysOnLog_Old.etl 2018-04-06 08:24:51.502775  1.548450e+09        NaN        NaN                         NaN                         NaN
3467     51.0  HxCommAlwaysOnLog_Old.etl 2018-04-06 12:26:20.523847  1.549958e+09        NaN        NaN                         NaN                         NaN
3471     51.0  HxCommAlwaysOnLog_Old.etl 2018-04-06 12:26:21.945348  1.550147e+09        NaN        NaN  2018-04-06 12:26:20.522848  2018-04-06 12:26:

## Step 8: Export Grouped Events

Save the grouped events to CSV for Phase 3 processing.

In [12]:
# [Cell 21] Export Grouped Events to CSV

# Export to CSV
df_grouped.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')

# Verify export
output_size_mb = OUTPUT_PATH.stat().st_size / (1024 * 1024)

print(f"=== Export Complete ===")
print(f"Output file: {OUTPUT_PATH}")
print(f"File size: {output_size_mb:.2f} MB")
print(f"Total records: {len(df_grouped):,}")
print(f"Columns: {len(df_grouped.columns)}")

# Column list for reference
print(f"\nColumn names:")
for i, col in enumerate(df_grouped.columns, 1):
    print(f"  {i:2d}. {col}")


=== Export Complete ===
Output file: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 2: Data Preprocessing & Event Grouping/Test Notebook Outputs/grouped_events_LoneWolf.csv
File size: 114.55 MB
Total records: 481,400
Columns: 29

Column names:
   1. dataID
   2. FileFRN
   3. FileName
   4. FilePath
   5. EventSource
   6. EventTimestamp
   7. LSN
   8. USN
   9. RedoOP
  10. UndoOP
  11. RedoOPName
  12. UndoOPName
  13. RecordOffset
  14. AttributeOffset
  15. TargetVCN
  16. IsTimestampChange
  17. Undo_$SI-C
  18. Undo_$SI-M
  19. Undo_$SI-E
  20. Undo_$SI-A
  21. Redo_$SI-C
  22. Redo_$SI-M
  23. Redo_$SI-E
  24. Redo_$SI-A
  25. ReasonCode
  26. ReasonFlags
  27. HasBasicInfoChange
  28. HasClose
  29. HasFileCreate


## Summary

### Phase 2 Processing Complete for LoneWolf Dataset

**Input Files:**
- LoneWolf-LogFile.csv
- LoneWolf-MFT.csv  
- LoneWolf-UsnJrnl.csv

**Output File:**
- grouped_events_LoneWolf.csv

### Key Statistics
- Events are grouped by FileFRN and ordered by timestamp
- LogFile and UsnJrnl events are unified into a single event stream
- All timestamps preserve nanosecond precision
- dataID column added for dataset tracking

### Ready for Phase 3
The grouped_events_LoneWolf.csv is now ready for:
- Feature engineering (Phase 3)
- Application of Oh et al. detection logic
- Labeling using Suspicious.csv ground truth


In [13]:
# [Cell 23] Final Statistics

print("=" * 60)
print("PHASE 2 COMPLETE: LoneWolf Dataset")
print("=" * 60)

print(f"\n{'Metric':<40} {'Value':>15}")
print("-" * 60)
print(f"{'Total events processed':<40} {len(df_grouped):>15,}")
print(f"{'Unique files with events':<40} {df_grouped['FileFRN'].nunique():>15,}")
print(f"{'LogFile events':<40} {len(df_logfile_final):>15,}")
print(f"{'UsnJrnl events':<40} {len(df_usnjrnl_final):>15,}")
print(f"{'Timestamp change events':<40} {len(ts_change_events):>15,}")
print(f"{'BASIC_INFO_CHANGE events':<40} {len(basic_info_events):>15,}")
print(f"{'Output file size (MB)':<40} {output_size_mb:>15.2f}")
print("-" * 60)

print("\nNext Step: Phase 3 - Feature Engineering & Forensic Logic Application")


PHASE 2 COMPLETE: LoneWolf Dataset

Metric                                             Value
------------------------------------------------------------
Total events processed                           481,400
Unique files with events                          17,897
LogFile events                                   128,551
UsnJrnl events                                   352,849
Timestamp change events                           20,373
BASIC_INFO_CHANGE events                              42
Output file size (MB)                             114.55
------------------------------------------------------------

Next Step: Phase 3 - Feature Engineering & Forensic Logic Application
